In [1]:
#| default_exp perf.profiling
#| export

import sys
import os
import time
import numpy as np
import tracemalloc
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict
import gc

# Import from TinyTorch package (previous modules must be completed and exported)
from tinytorch.core.tensor import Tensor
from tinytorch.core.layers import Linear
from tinytorch.core.spatial import Conv2d

# Constants for memory and performance measurement
BYTES_PER_FLOAT32 = 4  # Standard float32 size in bytes
KB_TO_BYTES = 1024  # Kilobytes to bytes conversion
MB_TO_BYTES = 1024 * 1024  # Megabytes to bytes conversion

In [20]:
#| export
class Profiler:
    """
    Professional-grade ML model profiler for performance analysis.

    Measures parameters, FLOPs, memory usage, and latency with statistical rigor.
    Used for optimization guidance and deployment planning.
    """

    def __init__(self):
        """
        Initialize profiler with measurement state.

        TODO: Set up profiler tracking structures

        APPROACH:
        1. Create empty measurements dictionary
        2. Initialize operation counters
        3. Set up memory tracking state

        EXAMPLE:
        >>> profiler = Profiler()
        >>> profiler.measurements
        {}

        HINTS:
        - Use defaultdict(int) for operation counters
        - measurements dict will store timing results
        """
        ### BEGIN SOLUTION
        self.measurements = {}
        self.operation_counts = defaultdict(int)
        self.memory_tracker = None
        ### END SOLUTION

    def _count_layer_parameters(self, layer) -> int:
        """
        Count parameters in a single layer by inspecting weight and bias attributes.

        Handles the fundamental unit of parameter counting: a single layer
        with weight and optional bias tensors.

        ```
        Single Layer Parameter Count:
        ┌─────────────────────────────────────────┐
        │ layer.weight.data.size  (e.g., 128×64)  │
        │ + layer.bias.data.size  (e.g., 64)      │
        │ = total layer parameters (e.g., 8256)    │
        └─────────────────────────────────────────┘
        ```

        Args:
            layer: A layer object with .weight (and optionally .bias)

        Returns:
            int: Total parameter count for this layer
        """
        ### BEGIN SOLUTION
        params = 0
        if hasattr(layer, "weight"):
            params += layer.weight.data.size
            if hasattr(layer, 'bias') and layer.bias is not None:
                params += layer.bias.data.size
        return params
        ### END SOLUTION

    def count_parameters(self, model) -> int:
        """
        Count total trainable parameters in a model.

        TODO: Implement parameter counting for any model with parameters() method

        APPROACH:
        1. Get all parameters from model.parameters() if available
        2. For single layers, use _count_layer_parameters() helper
        3. Sum total element count across all parameter tensors

        EXAMPLE:
        >>> linear = Linear(128, 64)  # 128*64 + 64 = 8256 parameters
        >>> profiler = Profiler()
        >>> count = profiler.count_parameters(linear)
        >>> print(count)
        8256

        HINTS:
        - Use _count_layer_parameters() for single layers
        - Use parameter.data.size for tensor element count
        - Handle models with and without parameters() method
        """
        ### BEGIN SOLUTION
        if hasattr(model, 'layers'):
            return sum(p.data.size for layer in model.layers for p in layer.parameters())
        elif hasattr(model, 'parameters'):
            return sum(p.data.size for p in model.parameters())
        elif hasattr(model, 'weight'):
            return self._count_layer_parameters(model)
        return 0
        ### END SOLUTION

    def _count_linear_flops(self, model, input_shape: Tuple[int, ...]) -> int:
        """
        Count FLOPs for a Linear layer forward pass.

        ```
        Linear FLOP Formula:
        FLOPs = in_features × out_features × 2
                     ↑              ↑          ↑
              Input dimension  Output dimension  Multiply + Add
        ```

        Args:
            model: A Linear layer with .weight attribute
            input_shape: Input tensor shape (batch, in_features)

        Returns:
            int: FLOP count for one forward pass (batch-independent)
        """
        ### BEGIN SOLUTION
        in_features = input_shape[-1]
        out_features = model.weight.shape[1] if hasattr(model, 'weight') else 1
        return in_features * out_features * 2
        ### END SOLUTION

    def _count_conv_flops(self, model, input_shape: Tuple[int, ...]) -> int:
        """
        Count FLOPs for a Conv2d layer forward pass.

        ```
        Conv2d FLOP Formula:
        FLOPs = out_H × out_W × kernel_H × kernel_W × in_C × out_C × 2
                  ↑       ↑        ↑          ↑         ↑       ↑      ↑
              Output spatial    Kernel spatial     Channel dims   Mul+Add
        ```

        Args:
            model: A Conv2d layer with kernel_size, in_channels, out_channels
            input_shape: Input tensor shape (batch, channels, height, width)

        Returns:
            int: FLOP count for one forward pass
        """
        ### BEGIN SOLUTION
        if not (hasattr(model, 'kernel_size') and hasattr(model, 'in_channels')):
            return 0
        
        in_channels = model.in_channels
        out_channels = model.out_channels
        kernel_h = kernel_w = model.kernel_size

        input_h, input_w = input_shape[-2], input_shape[-1]
        stride = model.stride if hasattr(model, 'stride') else 1
        output_h = input_h // stride
        output_w = input_w // stride

        return output_h * output_w * kernel_h * kernel_w * in_channels * out_channels * 2
        ### END SOLUTION

    def _count_sequential_flops(self, model, input_shape: Tuple[int, ...]) -> int:
        """
        Count FLOPs for a Sequential model by summing per-layer FLOPs.

        ```
        Sequential FLOP Accumulation:
        Layer 1 FLOPs + Layer 2 FLOPs + ... + Layer N FLOPs = Total FLOPs
             ↓               ↓                    ↓
          Shape propagated through each layer
        ```

        Args:
            model: A model with .layers attribute (list of layers)
            input_shape: Input tensor shape for the first layer

        Returns:
            int: Total FLOP count across all layers
        """
        ### BEGIN SOLUTION
        total_flops = 0
        current_shape = input_shape
        for layer in model.layers:
            total_flops += self.count_flops(layer, current_shape)
            if hasattr(layer, 'weight'):
                current_shape = current_shape[:-1] + (layer.weight.shape[1],)
        return total_flops
        ### END SOLUTION

    def count_flops(self, model, input_shape: Tuple[int, ...]) -> int:
        """
        Count FLOPs (Floating Point Operations) for one forward pass.

        TODO: Implement FLOP counting by dispatching to per-layer-type helpers

        APPROACH:
        1. Identify model type by class name
        2. Dispatch to _count_linear_flops, _count_conv_flops, or _count_sequential_flops
        3. Fall back to 1 FLOP per element for activations

        EXAMPLE:
        >>> linear = Linear(128, 64)
        >>> profiler = Profiler()
        >>> flops = profiler.count_flops(linear, (1, 128))
        >>> print(flops)  # 128 * 64 * 2 = 16384
        16384

        HINT: Use model.__class__.__name__ to identify layer type
        """
        ### BEGIN SOLUTION
        model_name = model.__class__.__name__

        if model_name == 'Linear':
            return self._count_linear_flops(model, input_shape)
        elif model_name == 'Conv2d':
            return self._count_conv_flops(model, input_shape)
        elif model_name == 'Sequential' or hasattr(model, 'layers'):
            return self._count_sequential_flops(model, input_shape)
        else:
            return int(np.prod(input_shape))
        ### END SOLUTION

    def _calculate_parameter_memory(self, model) -> float:
        """
        Calculate memory used by model parameters in megabytes.

        ```
        Parameter Memory Formula:
        Memory (MB) = parameter_count × 4 bytes / (1024 × 1024)
                           ↑              ↑
                     From count_parameters  FP32 size
        ```

        Args:
            model: Model to analyze

        Returns:
            float: Parameter memory in megabytes
        """
        ### BEGIN SOLUTION
        param_count = self.count_parameters(model)
        return (param_count * BYTES_PER_FLOAT32) / MB_TO_BYTES
        ### END SOLUTION

    def _calculate_memory_efficiency(self, useful_memory_mb: float, peak_memory_mb: float) -> float:
        """
        Calculate memory efficiency as ratio of useful to total memory.

        ```
        Efficiency = useful_memory / peak_memory
                         ↑               ↑
              Parameters + Activations   tracemalloc peak

        Ideal: 1.0 (all memory is useful)
        Typical: 0.3-0.8 (overhead from allocator, fragmentation)
        ```

        Args:
            useful_memory_mb: Sum of parameter + activation memory
            peak_memory_mb: Peak memory observed by tracemalloc

        Returns:
            float: Efficiency ratio clamped to [0, 1]
        """
        ### BEGIN SOLUTION
        ratio = useful_memory_mb / max(peak_memory_mb, 0.001)
        return min(ratio, 1.0)
        ### END SOLUTION

    def measure_memory(self, model, input_shape: Tuple[int, ...]) -> Dict[str, float]:
        """
        Measure memory usage during forward pass.

        TODO: Implement memory tracking using tracemalloc and helper methods

        APPROACH:
        1. Use _calculate_parameter_memory() for parameter bytes
        2. Use tracemalloc to track peak allocation during forward pass
        3. Use _calculate_memory_efficiency() for efficiency ratio

        EXAMPLE:
        >>> linear = Linear(1024, 512)
        >>> profiler = Profiler()
        >>> memory = profiler.measure_memory(linear, (32, 1024))
        >>> print(f"Parameters: {memory['parameter_memory_mb']:.1f} MB")
        Parameters: 2.1 MB

        HINT: tracemalloc.start() / get_traced_memory() / stop() lifecycle
        """
        ### BEGIN SOLUTION
        tracemalloc.start()
        _baseline_memory = tracemalloc.get_traced_memory()[0]

        parameter_memory_mb = self._calculate_parameter_memory(model)

        dummy_input = Tensor(np.random.randn(*input_shape))
        activation_memory_mb = (dummy_input.data.nbytes * 2) / MB_TO_BYTES

        _ = model.forward(dummy_input)

        _current_memory, peak_memory = tracemalloc.get_traced_memory()
        peak_memory_mb = (peak_memory - _baseline_memory) / MB_TO_BYTES
        tracemalloc.stop()

        useful_memory = parameter_memory_mb + activation_memory_mb
        return {
            'parameter_memory_mb': parameter_memory_mb,
            'activation_memory_mb': activation_memory_mb,
            'peak_memory_mb': max(peak_memory_mb, useful_memory),
            'memory_efficiency': self._calculate_memory_efficiency(useful_memory, peak_memory_mb)
        }
        ### END SOLUTION

    def measure_latency(self, model, input_tensor, warmup: int = 10, iterations: int = 100) -> float:
        """
        Measure model inference latency with statistical rigor.

        TODO: Implement accurate latency measurement

        APPROACH:
        1. Run warmup iterations to stabilize performance
        2. Measure multiple iterations for statistical accuracy
        3. Calculate median latency to handle outliers
        4. Return latency in milliseconds

        PARAMETERS:
        - warmup: Number of warmup runs (default 10)
        - iterations: Number of measurement runs (default 100)

        EXAMPLE:
        >>> linear = Linear(128, 64)
        >>> input_tensor = Tensor(np.random.randn(1, 128))
        >>> profiler = Profiler()
        >>> latency = profiler.measure_latency(linear, input_tensor)
        >>> print(f"Latency: {latency:.2f} ms")
        Latency: 0.15 ms

        HINTS:
        - Use time.perf_counter() for high precision
        - Use median instead of mean for robustness against outliers
        - Handle different model interfaces (forward, __call__)
        """
        ### BEGIN SOLUTION
        # Warmup runs to stabilize performance
        for _ in range(warmup):
            _ = model.forward(input_tensor)

        # measurement runs
        times = []
        for _ in range(iterations):
            start_time = time.perf_counter()
            _ = model.forward(input_tensor)
            end_time = time.perf_counter()
            times.append((end_time - start_time) * 1000) # convert to milliseconds

        # calculate statistics - use median for robustness
        times = np.array(times)
        median_latency = np.median(times)

        return float(median_latency)
        ### END SOLUTION

    def profile_layer(self, layer, input_shape: Tuple[int, ...]) -> Dict[str, Any]:
        """
        Profile a single layer comprehensively.

        TODO: Implement layer-wise profiling

        APPROACH:
        1. Count parameters for this layer
        2. Count FLOPs for this layer
        3. Measure memory usage
        4. Measure latency
        5. Return comprehensive layer profile

        EXAMPLE:
        >>> linear = Linear(256, 128)
        >>> profiler = Profiler()
        >>> profile = profiler.profile_layer(linear, (32, 256))
        >>> print(f"Layer uses {profile['parameters']} parameters")
        Layer uses 32896 parameters

        HINTS:
        - Use existing profiler methods (count_parameters, count_flops, etc.)
        - Create dummy input for latency measurement
        - Include layer type information in profile
        """
        ### BEGIN SOLUTION
        # Create dummy input for latency measurement
        dummy_input = Tensor(np.random.randn(*input_shape))

        # gather all measurements
        params = self.count_parameters(layer)
        flops = self.count_flops(layer, input_shape)
        memory = self.measure_memory(layer, input_shape)
        latency = self.memory_latency(layer, dummy_input, warmup=3, iteractions=10)

        # compute derived metrics
        gflops_per_second = (flops / 1e9) / max(latency / 1000, 1e-6)

        return {
            'layer_type': layer.__class__.__name__,
            'parameters': params,
            'flops': flops,
            'latency_ms': latency,
            'gflops_per_second': gflops_per_second,
            **memory
        }
        ### END SOLUTION

    def _compute_derived_metrics(self, flops: int, latency_ms: float,
                                  peak_memory_mb: float) -> Dict[str, float]:
        """
        Compute throughput and efficiency metrics from raw measurements.

        ```
        Derived Metrics Pipeline:
        FLOPs + Latency → GFLOP/s (throughput)
        Memory + Latency → MB/s (bandwidth)
        GFLOP/s / Peak → Efficiency (utilization)
        ```

        Args:
            flops: Total floating point operations
            latency_ms: Measured latency in milliseconds
            peak_memory_mb: Peak memory usage in megabytes

        Returns:
            dict with gflops_per_second, memory_bandwidth_mbs, computational_efficiency
        """
        ### BEGIN SOLUTION
        latency_seconds = latency_ms / 1000.0
        gflops_per_second = (flops / 1e9) / max(latency_seconds, 1e-6)
        memory_bandwidth = peak_memory_mb / max(latency_seconds, 1e-6)
        theoretical_peak_gflops = 100.0
        computational_efficiency = min(gflops_per_second / theoretical_peak_gflops, 1.0)

        return {
            'gflops_per_second': gflops_per_second,
            'memory_bandwidth_mbs': memory_bandwidth,
            'computational_efficiency': computational_efficiency
        }
        ### END SOLUTION

    def _analyze_bottleneck(self, gflops_per_second: float,
                            memory_bandwidth_mbs: float) -> Dict[str, Any]:
        """
        Identify whether workload is memory-bound or compute-bound.

        ```
        Bottleneck Decision:
        If bandwidth >> GFLOP/s × 100 → Memory-bound (data movement dominates)
        Otherwise                      → Compute-bound (arithmetic dominates)
        ```

        Args:
            gflops_per_second: Compute throughput
            memory_bandwidth_mbs: Memory bandwidth in MB/s

        Returns:
            dict with is_memory_bound, is_compute_bound, bottleneck label
        """
        ### BEGIN SOLUTION
        is_memory_bound = memory_bandwidth_mbs > gflops_per_second * 100
        return {
            'is_memory_bound': is_memory_bound,
            'is_compute_bound': not is_memory_bound,
            'bottleneck': 'memory' if is_memory_bound else 'compute'
        }
        ### END SOLUTION

    def profile_forward_pass(self, model, input_tensor) -> Dict[str, Any]:
        """
        Comprehensive profiling of a model's forward pass.

        TODO: Gather measurements, then use _compute_derived_metrics and _analyze_bottleneck

        APPROACH:
        1. Gather raw measurements (parameters, FLOPs, memory, latency)
        2. Use _compute_derived_metrics() for throughput and efficiency
        3. Use _analyze_bottleneck() for bottleneck identification

        EXAMPLE:
        >>> model = Linear(256, 128)
        >>> input_data = Tensor(np.random.randn(32, 256))
        >>> profiler = Profiler()
        >>> profile = profiler.profile_forward_pass(model, input_data)
        >>> print(f"Throughput: {profile['gflops_per_second']:.2f} GFLOP/s")
        Throughput: 2.45 GFLOP/s

        HINT: Compose helper outputs with ** unpacking into return dict
        """
        ### BEGIN SOLUTION
        param_count = self.count_parameters(model)
        flops = self.count_flops(model, input_tensor.shape)
        memory_stats = self.measure_memory(model, input_tensor.shape)
        latency_ms = self.measure_latency(model, input_tensor, warmup=5, iterations=20)

        derived = self._compute_derived_metrics(flops, latency_ms, memory_stats['peak_memory_mb'])
        bottleneck = self._analyze_bottleneck(derived['gflops_per_second'],
                                              derived['memory_bandwidth_mbs'])
        
        return {
            'parameters': param_count, 'flops': flops, 'latency_ms': latency_ms,
            **memory_stats, **derived, **bottleneck
        }
        ### END SOLUTION

    def _estimate_backward_costs(self, forward_flops: int,
                                  forward_latency_ms: float) -> Dict[str, float]:
        """
        Estimate backward pass compute costs from forward pass measurements.

        ```
        Backward Pass Cost Estimation:
        Backward FLOPs   = Forward FLOPs × 2   (gradient computation)
        Backward Latency = Forward Latency × 2 (more complex operations)

        Why 2×? Each operation needs:
        1. Gradient w.r.t. weights (same cost as forward)
        2. Gradient w.r.t. inputs (same cost as forward)
        ```

        Args:
            forward_flops: FLOP count from forward pass
            forward_latency_ms: Latency from forward pass

        Returns:
            dict with backward_flops and backward_latency_ms
        """
        ### BEGIN SOLUTION
        return {
            'backward_flops': forward_flops * 2,
            'backward_latency_ms': forward_latency_ms * 2
        }
        ### END SOLUTION

    def _estimate_optimizer_memory(self, gradient_memory_mb: float) -> Dict[str, float]:
        """
        Estimate additional memory required by different optimizers.

        ```
        Optimizer Memory Requirements:
        ┌───────────┬────────────────────────────────────┐
        │ Optimizer │ Extra Memory                       │
        ├───────────┼────────────────────────────────────┤
        │ SGD       │ 0× (no state)                      │
        │ Adam      │ 2× gradient memory (m + v)         │
        │ AdamW     │ 2× gradient memory (m + v)         │
        └───────────┴────────────────────────────────────┘
        ```

        Args:
            gradient_memory_mb: Memory for gradient storage in MB

        Returns:
            dict mapping optimizer name to extra memory in MB
        """
        ### BEGIN SOLUTION
        return {
            'sgd': 0,
            'adam': gradient_memory_mb * 2,
            'adamw': gradient_memory_mb * 2
        }
        ### END SOLUTION

    def profile_backward_pass(self, model, input_tensor, _loss_fn=None) -> Dict[str, Any]:
        """
        Profile both forward and backward passes for training analysis.

        TODO: Use _estimate_backward_costs and _estimate_optimizer_memory helpers

        APPROACH:
        1. Profile forward pass with profile_forward_pass()
        2. Use _estimate_backward_costs() for backward FLOPs and latency
        3. Use _estimate_optimizer_memory() for optimizer memory estimates
        4. Combine into total training iteration metrics

        EXAMPLE:
        >>> model = Linear(128, 64)
        >>> input_data = Tensor(np.random.randn(16, 128))
        >>> profiler = Profiler()
        >>> profile = profiler.profile_backward_pass(model, input_data)
        >>> print(f"Training iteration: {profile['total_latency_ms']:.2f} ms")
        Training iteration: 0.45 ms

        HINT: Gradient memory equals parameter memory (one gradient per parameter)
        """
        ### BEGIN SOLUTION
        fwd = self.profile_forward_pass(model, input_tensor)
        bwd = self._estimate_backward_costs(fwd['flops'], fwd['latency_ms'])

        gradient_memory_mb = fwd['parameter_memory_mb']
        total_flops = fwd['flops'] + bwd['backward_flops']
        total_latency_ms = fwd['latency_ms'] + bwd['backward_latency_ms']
        total_memory_mb = fwd['parameter_memory_mb'] + fwd['activation_memory_mb'] + gradient_memory_mb

        return {
            'forward_flops': fwd['flops'],
            'forward_latency_ms': fwd['latency_ms'],
            'forward_memory_mb': fwd['peak_memory_mb'],
            **bwd,
            'gradient_memory_mb': gradient_memory_mb,
            'total_flops': total_flops,
            'total_latency_ms': total_latency_ms,
            'total_memory_mb': total_memory_mb,
            'total_gflops_per_second': (total_flops / 1e9) / (total_latency_ms / 1000.0),
            'optimizer_memory_estimates': self._estimate_optimizer_memory(gradient_memory_mb),
            'memory_efficiency': fwd['memory_efficiency'],
            'bottleneck': fwd['bottleneck']
        }
        ### END SOLUTION

In [21]:
#| export
def quick_profile(model, input_tensor, profiler=None):
    """
    Quick profiling function for immediate insights.

    Provides a simplified interface for profiling that displays key metrics
    in a student-friendly format.

    Args:
        model: Model to profile
        input_tensor: Input data for profiling
        profiler: Optional Profiler instance (creates new one if None)

    Returns:
        dict: Profile results with key metrics

    Example:
        >>> model = Linear(128, 64)
        >>> input_data = Tensor(np.random.randn(16, 128))
        >>> results = quick_profile(model, input_data)
        >>> # Displays formatted output automatically
    """
    if profiler is None:
        profiler = Profiler()

    profile = profiler.profile_forward_pass(model, input_tensor)

    # Display formatted results
    print("🧪 Quick Profile Results:")
    print(f"   Parameters: {profile['parameters']:,}")
    print(f"   FLOPs: {profile['flops']:,}")
    print(f"   Latency: {profile['latency_ms']:.2f} ms")
    print(f"   Memory: {profile['peak_memory_mb']:.2f} MB")
    print(f"   Bottleneck: {profile['bottleneck']}")
    print(f"   Efficiency: {profile['computational_efficiency']*100:.1f}%")

    return profile

In [22]:
#| export
def analyze_weight_distribution(model, percentiles=[10, 25, 50, 75, 90]):
    """
    Analyze weight distribution across layers.

    Helps understand how weights are distributed across layers.
    Useful for identifying patterns in parameter magnitudes.

    Args:
        model: Model to analyze
        percentiles: List of percentiles to compute

    Returns:
        dict: Weight distribution statistics

    Example:
        >>> model = Linear(512, 512)
        >>> stats = analyze_weight_distribution(model)
        >>> print(f"Weights < 0.01: {stats['below_threshold_001']:.1f}%")
    """
    # Collect all weights
    weights = []
    if hasattr(model, 'parameters'):
        for param in model.parameters():
            weights.extend(param.data.flatten().tolist())
    elif hasattr(model, 'weight'):
        weights.extend(model.weight.data.flatten().tolist())
    else:
        return {'error': 'No weights found'}

    weights = np.array(weights)
    abs_weights = np.abs(weights)

    # Calculate statistics
    stats = {
        'total_weights': len(weights),
        'mean': float(np.mean(abs_weights)),
        'std': float(np.std(abs_weights)),
        'min': float(np.min(abs_weights)),
        'max': float(np.max(abs_weights)),
    }

    # Percentile analysis
    for p in percentiles:
        stats[f'percentile_{p}'] = float(np.percentile(abs_weights, p))

    # Threshold analysis (useful for pruning)
    for threshold in [0.001, 0.01, 0.1]:
        below = np.sum(abs_weights < threshold) / len(weights) * 100
        stats[f'below_threshold_{str(threshold).replace(".", "")}'] = below

    return stats

In [23]:
def test_unit_helper_functions():
    """🧪 Test helper function implementations."""
    print("🧪 Unit Test: Helper Functions...")

    # Test 1: Quick profile function
    from tinytorch.core.layers import Linear
    test_model = Linear(16, 8)
    test_input = Tensor(np.random.randn(8, 16))
    profile = quick_profile(test_model, test_input, profiler=Profiler())

    # Validate profile contains expected keys
    assert 'parameters' in profile, "Quick profile should include parameters"
    assert 'flops' in profile, "Quick profile should include FLOPs"
    assert 'latency_ms' in profile, "Quick profile should include latency"
    print("✅ Quick profile provides comprehensive metrics")

    # Test 2: Weight distribution analysis
    class SimpleModel:
        def __init__(self):
            self.weight = Tensor(np.random.randn(10, 5) * 0.1)  # Small weights

    model = SimpleModel()
    stats = analyze_weight_distribution(model)

    # Validate statistics structure
    assert 'total_weights' in stats, "Should count total weights"
    assert 'mean' in stats, "Should compute mean"
    assert 'std' in stats, "Should compute standard deviation"
    assert stats['total_weights'] == 50, f"Expected 50 weights, got {stats['total_weights']}"
    print(f"✅ Weight distribution analysis: {stats['total_weights']} weights analyzed")

    # Test 3: Weight distribution with no weights
    class NoWeightModel:
        pass

    no_weight_model = NoWeightModel()
    stats = analyze_weight_distribution(no_weight_model)
    assert 'error' in stats, "Should handle models without weights"
    print("✅ Handles models without weights gracefully")

    print("✅ Helper functions work correctly!")

if __name__ == "__main__":
    test_unit_helper_functions()

🧪 Unit Test: Helper Functions...
🧪 Quick Profile Results:
   Parameters: 136
   FLOPs: 256
   Latency: 0.15 ms
   Memory: 0.01 MB
   Bottleneck: memory
   Efficiency: 0.0%
✅ Quick profile provides comprehensive metrics
✅ Weight distribution analysis: 50 weights analyzed
✅ Handles models without weights gracefully
✅ Helper functions work correctly!


In [24]:
def test_unit_count_layer_parameters():
    """🧪 Test _count_layer_parameters helper."""
    print("🧪 Unit Test: _count_layer_parameters...")

    profiler = Profiler()

    # Test 1: Layer with weight and bias
    class LayerWithBias:
        def __init__(self):
            self.weight = Tensor(np.random.randn(10, 5))
            self.bias = Tensor(np.random.randn(5))

    layer = LayerWithBias()
    count = profiler._count_layer_parameters(layer)
    assert count == 55, f"Expected 55 (10*5 + 5), got {count}"
    print(f"✅ Layer with bias: {count} parameters")

    # Test 2: Layer with weight only (no bias)
    class LayerNoBias:
        def __init__(self):
            self.weight = Tensor(np.random.randn(8, 4))

    layer_no_bias = LayerNoBias()
    count = profiler._count_layer_parameters(layer_no_bias)
    assert count == 32, f"Expected 32 (8*4), got {count}"
    print(f"✅ Layer without bias: {count} parameters")

    # Test 3: Object without weight attribute
    class NoWeight:
        pass

    count = profiler._count_layer_parameters(NoWeight())
    assert count == 0, f"Expected 0, got {count}"
    print("✅ No weight attribute: 0 parameters")

    print("✅ _count_layer_parameters works correctly!")

if __name__ == "__main__":
    test_unit_count_layer_parameters()

🧪 Unit Test: _count_layer_parameters...
✅ Layer with bias: 55 parameters
✅ Layer without bias: 32 parameters
✅ No weight attribute: 0 parameters
✅ _count_layer_parameters works correctly!


In [25]:
def test_unit_parameter_counting():
    """🧪 Test parameter counting implementation."""
    print("🧪 Unit Test: Parameter Counting...")

    profiler = Profiler()

    # Test 1: Simple model with known parameters
    class SimpleModel:
        def __init__(self):
            self.weight = Tensor(np.random.randn(10, 5))
            self.bias = Tensor(np.random.randn(5))

        def parameters(self):
            return [self.weight, self.bias]

    simple_model = SimpleModel()
    param_count = profiler.count_parameters(simple_model)
    expected_count = 10 * 5 + 5  # weight + bias
    assert param_count == expected_count, f"Expected {expected_count} parameters, got {param_count}"
    print(f"✅ Simple model: {param_count} parameters")

    # Test 2: Model without parameters
    class NoParamModel:
        def __init__(self):
            pass

    no_param_model = NoParamModel()
    param_count = profiler.count_parameters(no_param_model)
    assert param_count == 0, f"Expected 0 parameters, got {param_count}"
    print(f"✅ No parameter model: {param_count} parameters")

    # Test 3: Direct tensor (no parameters)
    test_tensor = Tensor(np.random.randn(2, 3))
    param_count = profiler.count_parameters(test_tensor)
    assert param_count == 0, f"Expected 0 parameters for tensor, got {param_count}"
    print(f"✅ Direct tensor: {param_count} parameters")

    print("✅ Parameter counting works correctly!")

if __name__ == "__main__":
    test_unit_parameter_counting()

🧪 Unit Test: Parameter Counting...
✅ Simple model: 55 parameters
✅ No parameter model: 0 parameters
✅ Direct tensor: 0 parameters
✅ Parameter counting works correctly!


In [26]:
def test_unit_count_linear_flops():
    """🧪 Test _count_linear_flops helper."""
    print("🧪 Unit Test: _count_linear_flops...")

    profiler = Profiler()

    # Create mock Linear layer
    class MockLinear:
        def __init__(self, in_f, out_f):
            self.weight = Tensor(np.random.randn(in_f, out_f))
            self.__class__.__name__ = 'Linear'

    # Test 1: Known dimensions
    layer = MockLinear(128, 64)
    flops = profiler._count_linear_flops(layer, (1, 128))
    assert flops == 128 * 64 * 2, f"Expected {128*64*2}, got {flops}"
    print(f"✅ Linear(128, 64): {flops} FLOPs")

    # Test 2: Square layer
    layer_sq = MockLinear(256, 256)
    flops_sq = profiler._count_linear_flops(layer_sq, (1, 256))
    assert flops_sq == 256 * 256 * 2, f"Expected {256*256*2}, got {flops_sq}"
    print(f"✅ Linear(256, 256): {flops_sq} FLOPs")

    # Test 3: Batch independence (uses last dim only)
    flops_b1 = profiler._count_linear_flops(layer, (1, 128))
    flops_b32 = profiler._count_linear_flops(layer, (32, 128))
    assert flops_b1 == flops_b32, "FLOPs should be batch-independent"
    print("✅ Batch-independent FLOPs confirmed")

    print("✅ _count_linear_flops works correctly!")

if __name__ == "__main__":
    test_unit_count_linear_flops()

🧪 Unit Test: _count_linear_flops...
✅ Linear(128, 64): 16384 FLOPs
✅ Linear(256, 256): 131072 FLOPs
✅ Batch-independent FLOPs confirmed
✅ _count_linear_flops works correctly!


In [27]:
def test_unit_count_conv_flops():
    """🧪 Test _count_conv_flops helper."""
    print("🧪 Unit Test: _count_conv_flops...")

    profiler = Profiler()

    # Create mock Conv2d layer
    class MockConv:
        def __init__(self, in_c, out_c, k, s=1):
            self.in_channels = in_c
            self.out_channels = out_c
            self.kernel_size = k
            self.stride = s
            self.__class__.__name__ = 'Conv2d'

    # Test 1: Simple 3x3 conv, stride 1
    conv = MockConv(3, 16, 3, 1)
    flops = profiler._count_conv_flops(conv, (1, 3, 32, 32))
    expected = 32 * 32 * 3 * 3 * 3 * 16 * 2
    assert flops == expected, f"Expected {expected}, got {flops}"
    print(f"✅ Conv2d(3, 16, 3): {flops} FLOPs")

    # Test 2: Stride 2 halves output spatial dims
    conv_s2 = MockConv(3, 64, 7, 2)
    flops_s2 = profiler._count_conv_flops(conv_s2, (1, 3, 224, 224))
    out_h, out_w = 224 // 2, 224 // 2
    expected_s2 = out_h * out_w * 7 * 7 * 3 * 64 * 2
    assert flops_s2 == expected_s2, f"Expected {expected_s2}, got {flops_s2}"
    print(f"✅ Conv2d(3, 64, 7, stride=2): {flops_s2} FLOPs")

    # Test 3: Missing attributes returns 0
    class Incomplete:
        pass

    assert profiler._count_conv_flops(Incomplete(), (1, 3, 32, 32)) == 0
    print("✅ Missing attributes returns 0")

    print("✅ _count_conv_flops works correctly!")

if __name__ == "__main__":
    test_unit_count_conv_flops()

🧪 Unit Test: _count_conv_flops...
✅ Conv2d(3, 16, 3): 884736 FLOPs
✅ Conv2d(3, 64, 7, stride=2): 236027904 FLOPs
✅ Missing attributes returns 0
✅ _count_conv_flops works correctly!


In [28]:
def test_unit_count_sequential_flops():
    """🧪 Test _count_sequential_flops helper."""
    print("🧪 Unit Test: _count_sequential_flops...")

    profiler = Profiler()

    # Create mock sequential model with two Linear layers
    class MockLinear:
        def __init__(self, in_f, out_f):
            self.weight = Tensor(np.random.randn(in_f, out_f))
            self.__class__.__name__ = 'Linear'

    class MockSequential:
        def __init__(self, *layer_list):
            self.layers = list(layer_list)

    model = MockSequential(MockLinear(128, 64), MockLinear(64, 10))
    total_flops = profiler._count_sequential_flops(model, (1, 128))

    expected = (128 * 64 * 2) + (64 * 10 * 2)
    assert total_flops == expected, f"Expected {expected}, got {total_flops}"
    print(f"✅ Sequential(128->64->10): {total_flops} FLOPs")

    # Single layer sequential
    model_single = MockSequential(MockLinear(32, 16))
    flops_single = profiler._count_sequential_flops(model_single, (1, 32))
    assert flops_single == 32 * 16 * 2, f"Expected {32*16*2}, got {flops_single}"
    print(f"✅ Single-layer sequential: {flops_single} FLOPs")

    print("✅ _count_sequential_flops works correctly!")

if __name__ == "__main__":
    test_unit_count_sequential_flops()

🧪 Unit Test: _count_sequential_flops...
✅ Sequential(128->64->10): 17664 FLOPs
✅ Single-layer sequential: 1024 FLOPs
✅ _count_sequential_flops works correctly!


In [29]:
def test_unit_flop_counting():
    """🧪 Test FLOP counting implementation."""
    print("🧪 Unit Test: FLOP Counting...")

    profiler = Profiler()

    # Test 1: Simple tensor operations
    test_tensor = Tensor(np.random.randn(4, 8))
    flops = profiler.count_flops(test_tensor, (4, 8))
    expected_flops = 4 * 8  # 1 FLOP per element for generic operation
    assert flops == expected_flops, f"Expected {expected_flops} FLOPs, got {flops}"
    print(f"✅ Tensor operation: {flops} FLOPs")

    # Test 2: Simulated Linear layer
    class MockLinear:
        def __init__(self, in_features, out_features):
            self.weight = Tensor(np.random.randn(in_features, out_features))
            self.__class__.__name__ = 'Linear'

    mock_linear = MockLinear(128, 64)
    flops = profiler.count_flops(mock_linear, (1, 128))
    expected_flops = 128 * 64 * 2  # matmul FLOPs
    assert flops == expected_flops, f"Expected {expected_flops} FLOPs, got {flops}"
    print(f"✅ Linear layer: {flops} FLOPs")

    # Test 3: Batch size independence
    flops_batch1 = profiler.count_flops(mock_linear, (1, 128))
    flops_batch32 = profiler.count_flops(mock_linear, (32, 128))
    assert flops_batch1 == flops_batch32, "FLOPs should be independent of batch size"
    print(f"✅ Batch independence: {flops_batch1} FLOPs (same for batch 1 and 32)")

    print("✅ FLOP counting works correctly!")

if __name__ == "__main__":
    test_unit_flop_counting()

🧪 Unit Test: FLOP Counting...
✅ Tensor operation: 32 FLOPs
✅ Linear layer: 16384 FLOPs
✅ Batch independence: 16384 FLOPs (same for batch 1 and 32)
✅ FLOP counting works correctly!


In [30]:
def test_unit_calculate_parameter_memory():
    """🧪 Test _calculate_parameter_memory helper."""
    print("🧪 Unit Test: _calculate_parameter_memory...")

    profiler = Profiler()

    # Test 1: Known parameter count -> known memory
    class KnownModel:
        def __init__(self):
            # 1024 * 1024 = 1,048,576 parameters = exactly 4 MB at FP32
            self.weight = Tensor(np.random.randn(1024, 1024))

    model = KnownModel()
    memory_mb = profiler._calculate_parameter_memory(model)
    expected_mb = (1024 * 1024 * 4) / (1024 * 1024)  # 4.0 MB
    assert abs(memory_mb - expected_mb) < 0.01, f"Expected {expected_mb} MB, got {memory_mb}"
    print(f"✅ 1M params = {memory_mb:.1f} MB")

    # Test 2: Zero parameter model
    class EmptyModel:
        pass

    empty_mb = profiler._calculate_parameter_memory(EmptyModel())
    assert empty_mb == 0.0, f"Expected 0.0 MB, got {empty_mb}"
    print("✅ Empty model = 0.0 MB")

    print("✅ _calculate_parameter_memory works correctly!")

if __name__ == "__main__":
    test_unit_calculate_parameter_memory()

🧪 Unit Test: _calculate_parameter_memory...
✅ 1M params = 4.0 MB
✅ Empty model = 0.0 MB
✅ _calculate_parameter_memory works correctly!


In [31]:
def test_unit_calculate_memory_efficiency():
    """🧪 Test _calculate_memory_efficiency helper."""
    print("🧪 Unit Test: _calculate_memory_efficiency...")

    profiler = Profiler()

    # Test 1: Perfect efficiency
    eff = profiler._calculate_memory_efficiency(10.0, 10.0)
    assert abs(eff - 1.0) < 0.01, f"Expected 1.0, got {eff}"
    print(f"✅ Perfect efficiency: {eff}")

    # Test 2: Half efficiency
    eff_half = profiler._calculate_memory_efficiency(5.0, 10.0)
    assert abs(eff_half - 0.5) < 0.01, f"Expected 0.5, got {eff_half}"
    print(f"✅ Half efficiency: {eff_half}")

    # Test 3: Clamped at 1.0 (useful > peak shouldn't exceed 1.0)
    eff_clamped = profiler._calculate_memory_efficiency(20.0, 10.0)
    assert eff_clamped <= 1.0, f"Efficiency should be clamped to 1.0, got {eff_clamped}"
    print(f"✅ Clamped efficiency: {eff_clamped}")

    # Test 4: Division by zero safety
    eff_zero = profiler._calculate_memory_efficiency(5.0, 0.0)
    assert eff_zero <= 1.0, f"Should handle zero peak safely, got {eff_zero}"
    print("✅ Zero-peak safety handled")

    print("✅ _calculate_memory_efficiency works correctly!")

if __name__ == "__main__":
    test_unit_calculate_memory_efficiency()

🧪 Unit Test: _calculate_memory_efficiency...
✅ Perfect efficiency: 1.0
✅ Half efficiency: 0.5
✅ Clamped efficiency: 1.0
✅ Zero-peak safety handled
✅ _calculate_memory_efficiency works correctly!


In [32]:
def test_unit_memory_measurement():
    """🧪 Test memory measurement implementation."""
    print("🧪 Unit Test: Memory Measurement...")

    profiler = Profiler()

    # Test 1: Basic memory measurement
    test_tensor = Tensor(np.random.randn(10, 20))
    from tinytorch.core.layers import Linear
    test_model = Linear(20, 10)
    memory_stats = profiler.measure_memory(test_model, (10, 20))

    # Validate dictionary structure
    required_keys = ['parameter_memory_mb', 'activation_memory_mb', 'peak_memory_mb', 'memory_efficiency']
    for key in required_keys:
        assert key in memory_stats, f"Missing key: {key}"

    # Validate non-negative values
    for key in required_keys:
        assert memory_stats[key] >= 0, f"{key} should be non-negative, got {memory_stats[key]}"

    print(f"✅ Basic measurement: {memory_stats['peak_memory_mb']:.3f} MB peak")

    # Test 2: Memory scaling with size
    from tinytorch.core.layers import Linear
    small_model = Linear(5, 5)
    large_model = Linear(50, 50)

    small_memory = profiler.measure_memory(small_model, (5, 5))
    large_memory = profiler.measure_memory(large_model, (50, 50))

    # Larger tensor should use more activation memory
    assert large_memory['activation_memory_mb'] >= small_memory['activation_memory_mb'], \
        "Larger tensor should use more activation memory"

    print(f"✅ Scaling: Small {small_memory['activation_memory_mb']:.3f} MB → Large {large_memory['activation_memory_mb']:.3f} MB")

    # Test 3: Efficiency bounds
    assert 0 <= memory_stats['memory_efficiency'] <= 1.0, \
        f"Memory efficiency should be between 0 and 1, got {memory_stats['memory_efficiency']}"

    print(f"✅ Efficiency: {memory_stats['memory_efficiency']:.3f} (0-1 range)")

    print("✅ Memory measurement works correctly!")

if __name__ == "__main__":
    test_unit_memory_measurement()

🧪 Unit Test: Memory Measurement...
✅ Basic measurement: 0.003 MB peak
✅ Scaling: Small 0.000 MB → Large 0.019 MB
✅ Efficiency: 0.669 (0-1 range)
✅ Memory measurement works correctly!
